# Deliverable D4: Natural Language Request Understanding & Slot Extraction

This notebook implements and evaluates the conversational request understanding subsystem for SmartCart:
1. **Calibrated Baseline Intent Classifier**: TF-IDF vectorization paired with calibrated LinearSVC.
2. **Transformer Architecture**: DistilBERT sequence classification (intent) and token classification (BIO slot spans).
3. **Slot Normalization & Extraction**: Dictionary/regex rules, BIO span decoding, and typed normalization (budget, servings, dietary constraints).
4. **Explicit Uncertainty Gating**: Confidence calibration thresholds, missing entity clarification, and unsupported request routing.
5. **Empirical Evaluation**: Intent Macro-F1, confusion matrix, slot extraction accuracy, and qualitative review of validation failures.


In [ ]:
# Environment setup and modular NLU imports
os=__import__('os')
sys=__import__('sys')
json=__import__('json')
Path=__import__('pathlib').Path
plt=__import__('matplotlib.pyplot',fromlist=['pyplot'])
np=__import__('numpy')
torch=__import__('torch')
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'
sys.path.insert(0,str(Path.cwd()/'src'))
from	smartcart.nlu.dataset	import	generate_nlu_training_examples
from	smartcart.nlu.intent_model	import	train_intent_classifier,predict_intent_with_confidence
from	smartcart.nlu.transformer_model	import	build_distilbert_intent_model,predict_distilbert_intent,build_distilbert_slot_model,predict_distilbert_slots
from	smartcart.nlu.slots	import	extract_slots_rule_based
from	smartcart.nlu.spans	import	decode_bio_spans
from	smartcart.nlu.normalization	import	normalize_slot_spans
from	smartcart.nlu.uncertainty	import	check_request_clarification
from	smartcart.nlu.runner	import	execute_nlu_benchmark,generate_evaluation_benchmark_data
print('Modular	NLU	subsystem	loaded	successfully')


### 1. Ingest NLU Seed Training & Annotation Dataset
We inspect the seed conversational dataset containing annotated intents and token-level BIO slot annotations.


In [ ]:
# Load and inspect seed training examples
train_data=generate_nlu_training_examples()
print('Total	training	examples:',len(train_data))
for	i,ex	in	enumerate(train_data[:3]):
	print(f'Ex	{i+1}:',ex['text'],'->	Intent:',ex['intent'])


### 2. Calibrated Baseline Intent Classifier (TF-IDF + LinearSVC)
We train the baseline scikit-learn pipeline using sublinear TF-IDF n-grams and 2-fold cross-validated sigmoid probability calibration.


In [ ]:
# Train calibrated LinearSVC baseline pipeline
texts=[x['text']for	x	in	train_data]
labels=[x['intent']for	x	in	train_data]
baseline_pipe=train_intent_classifier(texts,labels)
print('Fitted	classes:',list(baseline_pipe.classes_))
pred=predict_intent_with_confidence(baseline_pipe,'cook	pasta	under	500')
print('Prediction:',pred['intent'],'Confidence:',round(pred['confidence'],4))


### 3. DistilBERT Transformer Architecture
We instantiate the proposed fine-tunable DistilBERT transformer models for intent sequence classification and token-level slot tagging.


In [ ]:
# Initialize and test DistilBERT sequence and token architectures
distil_intent=build_distilbert_intent_model(num_labels=4)
distil_slots=build_distilbert_slot_model(num_labels=9)
dummy_ids=torch.tensor([[101,7592,102]])
intent_out=predict_distilbert_intent(distil_intent,dummy_ids)
slot_out=predict_distilbert_slots(distil_slots,dummy_ids)
print('DistilBERT	intent	pred:',intent_out['predicted_class'],'conf:',round(intent_out['confidence'],4))
print('DistilBERT	slot	tag	indices:',slot_out)


### 4. Slot Span Extraction & Typed Normalization
Rule-based regex patterns, BIO span decoding, and typed normalization extract structured slots (budget limit, servings, dietary constraints).


In [ ]:
# Demonstrate slot extraction, BIO span decoding, and typed normalization
test_query='cook	pasta	for	4	people	under	$50'
rule_slots=extract_slots_rule_based(test_query)
print('Rule-based	extracted	slots:',rule_slots)
bio_tokens=['cook','pasta','for','4','people']
bio_tags=['O','B-MEAL','O','B-SERVINGS','O']
decoded_spans=decode_bio_spans(bio_tokens,bio_tags)
print('Decoded	BIO	spans:',decoded_spans)
norm_slots=normalize_slot_spans([{'label':'BUDGET','text':'$50'},{'label':'SERVINGS','text':'4'}])
print('Normalized	slots:',norm_slots)


### 5. Explicit Uncertainty Detection & Confidence Gating
We evaluate conversational uncertainty handling: triggering clarifying questions when budgets, amounts, or recipes are ambiguous or when intent confidence is low.


In [ ]:
# Test uncertainty gating on ambiguous and out-of-scope requests
ambiguous_req={'intent':'plan_meal'}
clarify_plan=check_request_clarification(ambiguous_req)
print('Missing	recipe	clarification:',clarify_plan['needs_clarification'],clarify_plan['prompt'])
pantry_req={'intent':'build_grocery_list','pantry':{'rice':None}}
clarify_pantry=check_request_clarification(pantry_req)
print('Pantry	quantity	clarification:',clarify_pantry['needs_clarification'],clarify_pantry['prompt'])
low_conf_req={'intent':'plan_meal'}
clarify_conf=check_request_clarification(low_conf_req,confidence=0.35,confidence_threshold=0.6)
print('Low	confidence	clarification:',clarify_conf['needs_clarification'],clarify_conf['prompt'])


### 6. Comprehensive NLU Benchmark Evaluation & Validation Failure Analysis
We run the validation benchmark comparing model predictions, reporting Macro-F1, confusion matrix, slot accuracies, and reviewing failure modes across 30 test scenarios.


In [ ]:
# Run benchmark and display evaluation summary
eval_results=execute_nlu_benchmark('results/predictions/nlu_evaluation_results.json','results/predictions/nlu_validation_failures.json')
print('NLU	Intent	Accuracy:',round(eval_results['intent_metrics']['accuracy'],4))
print('NLU	Intent	Macro-F1:',round(eval_results['intent_metrics']['macro_f1'],4))
print('Budget	Normalization	Accuracy:',round(eval_results['slot_metrics']['budget_accuracy'],4))
print('Servings	Normalization	Accuracy:',round(eval_results['slot_metrics']['servings_accuracy'],4))
print('Validation	failures	analyzed:',eval_results['num_failures_analyzed'])


### 7. Confusion Matrix & Training Curve Visualizations
We render the multi-class confusion matrix and learning curve comparison generated during evaluation.


In [ ]:
# Render confusion matrix and model comparison figures
img1=plt.imread('results/figures/09_nlu_intent_confusion_matrix.png')
fig=plt.figure(figsize=(7,5),dpi=150)
plt.imshow(img1)
plt.axis('off')
plt.title('NLU	Intent	Confusion	Matrix')
plt.show()
img2=plt.imread('results/figures/10_nlu_training_and_comparison_curves.png')
fig2=plt.figure(figsize=(8,5),dpi=150)
plt.imshow(img2)
plt.axis('off')
plt.title('DistilBERT	vs	Baseline	Learning	Curves')
plt.show()
